In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("bolt_andmed.csv") ## andmestiku sisselugemine

In [ ]:
df.head() ## esimesed 5 rida    

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_token,rider_app_version,order_state,order_try_state,driver_app_version,driver_device_uid_new,device_name,eu_indicator,overpaid_ride_ticket,fraud_score
0,22,22,2020-02-02 3:37:31,4.04,10.0,2839,700,1,client,finished,...,NaN,CI.4.17,finished,finished,DA.4.37,1596,Xiaomi Redmi 6,1,0,-1383.0
1,618,618,2020-02-08 2:26:19,6.09,3.6,5698,493,1,client,finished,...,NaN,CA.5.43,finished,finished,DA.4.39,1578,Samsung SM-G965F,1,0,NaN
2,657,657,2020-02-08 11:50:35,4.32,3.5,4426,695,1,client,finished,...,NaN,CA.5.43,finished,finished,DA.4.37,951,Samsung SM-A530F,1,0,-166.0
3,313,313,2020-02-05 6:34:54,72871.72,NaN,49748,1400,0,client,finished,...,NaN,CA.5.23,finished,finished,DA.4.37,1587,TECNO-Y6,0,1,NaN
4,1176,1176,2020-02-13 17:31:24,20032.50,19500.0,10273,5067,1,client,finished,...,NaN,CA.5.04,finished,finished,DA.4.37,433,Itel W5504,0,0,NaN


In [ ]:
df_clean = df.copy() ## teeme koopia, tötame edasi df_cleaniga

## Andmestiku info ja väljade kontroll

In [ ]:
df_clean.info() ## andmestiku info

<class 'pandas.DataFrame'>
RangeIndex: 4943 entries, 0 to 4942
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id_new           4943 non-null   int64  
 1   order_try_id_new       4943 non-null   int64  
 2   calc_created           4943 non-null   str    
 3   metered_price          4923 non-null   float64
 4   upfront_price          3409 non-null   float64
 5   distance               4943 non-null   int64  
 6   duration               4943 non-null   int64  
 7   gps_confidence         4943 non-null   int64  
 8   entered_by             4943 non-null   str    
 9   b_state                4943 non-null   str    
 10  dest_change_number     4943 non-null   int64  
 11  prediction_price_type  4923 non-null   str    
 12  predicted_distance     4923 non-null   float64
 13  predicted_duration     4923 non-null   float64
 14  change_reason_pricing  298 non-null    str    
 15  ticket_id_new  

In [8]:
df_clean["calc_created"] = pd.to_datetime(df_clean["calc_created"]) ## muudame kuupäeva õigeks tüübiks

In [9]:
df_clean["calc_created"].dtype ## seejärel kontrollin, kas on õige tüüp nüüd

dtype('<M8[us]')

## Kontrollime puuduvaid väärtuseid

In [ ]:
df_clean.isnull().sum() ## kontrollime puuduvaid väärtuseid


order_id_new                0
order_try_id_new            0
calc_created                0
metered_price              20
upfront_price            1534
distance                    0
duration                    0
gps_confidence              0
entered_by                  0
b_state                     0
dest_change_number          0
prediction_price_type      20
predicted_distance         20
predicted_duration         20
change_reason_pricing    4645
ticket_id_new               0
device_token             4943
rider_app_version          16
order_state                 0
order_try_state             0
driver_app_version          0
driver_device_uid_new       0
device_name                 0
eu_indicator                0
overpaid_ride_ticket        0
fraud_score              2759
dtype: int64

In [11]:
df_clean = df_clean.drop(columns=["device_token"])  ## eemaldan täiesti tühja veeru

In [12]:
df_clean.shape ## vaatame kui suur table nüüd on 

(4943, 25)

## Eraldame kuupäeva välja

In [21]:
## puhastame välja aasta, kuu, nädalapäeva ja tunni 

df_clean["date"] = df_clean["calc_created"].dt.date
df_clean["time"] = df_clean["calc_created"].dt.time

df_clean["day_of_week"] = df_clean["calc_created"].dt.day_name()
df_clean["day_of_week_nr"] = df_clean["calc_created"].dt.dayofweek + 1

df_clean["month"] = df_clean["calc_created"].dt.month
df_clean["year"] = df_clean["calc_created"].dt.year

In [22]:
df_clean[
    ["calc_created", "date", "time", "day_of_week",
     "day_of_week_nr", "month", "year"]
].head() 

## kontrollime kas eraldas kuupäeva

,calc_created,date,time,day_of_week,day_of_week_nr,month,year
0,2020-02-02 03:37:31,2020-02-02,03:37:31,Sunday,7,2,2020
1,2020-02-08 02:26:19,2020-02-08,02:26:19,Saturday,6,2,2020
2,2020-02-08 11:50:35,2020-02-08,11:50:35,Saturday,6,2,2020
3,2020-02-05 06:34:54,2020-02-05,06:34:54,Wednesday,3,2,2020
4,2020-02-13 17:31:24,2020-02-13,17:31:24,Thursday,4,2,2020


In [38]:
# Eemaldan varasemast katsest jäänud "hour" veeru,
# sest kasutan selle asemel täpset kellaaega "time"

df_clean = df_clean.drop(columns=["hour"])

In [41]:
# Eemaldan "weekday" veeru, sest "day_of_week" sisaldab täpselt sama infot
# ja jätan alles selgema nimega veeru Power BI jaoks

df_clean = df_clean.drop(columns=["weekday"])

In [42]:
# Kontrollin, kas Power BI jaoks loodud ajaveerud on kõik olemas
time_columns = ["date", "time", "day_of_week", "day_of_week_nr", "month", "year"]

[col for col in time_columns if col in df_clean.columns]

['date', 'time', 'day_of_week', 'day_of_week_nr', 'month', 'year']

## Kontrollin kas on identseid ridasid

In [24]:
df_clean.duplicated().sum()  ## kontrollin, mitu täielikult identset rida on

np.int64(0)

### Kui palju kordub tellimusi

In [25]:
df_clean["order_id_new"].duplicated().sum() ## kontrollin kui palju kordub tellimusi

np.int64(777)

#### Kuva korduvate ID'de read

In [27]:
## kuva kõik korduvate order ID-de read

duplicate_orders = df_clean[
    df_clean["order_id_new"].duplicated(keep=False)
]

duplicate_orders.sort_values("order_id_new").head(20)

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,overpaid_ride_ticket,fraud_score,year,month,weekday,hour,date,time,day_of_week,day_of_week_nr
1532,3,3,2020-02-02 00:49:24,14.87,NaN,15541,1690,0,client,finished,...,0,-1516.0,2020,2,Sunday,0,2020-02-02,00:49:24,Sunday,7
1902,3,3,2020-02-02 00:49:24,14.87,NaN,15541,1690,0,client,finished,...,0,-1516.0,2020,2,Sunday,0,2020-02-02,00:49:24,Sunday,7
2299,13,13,2020-02-02 02:31:56,8.89,NaN,16880,1339,1,driver,finished,...,0,-196.0,2020,2,Sunday,2,2020-02-02,02:31:56,Sunday,7
966,13,13,2020-02-02 02:31:56,8.89,NaN,16880,1339,1,driver,finished,...,0,-196.0,2020,2,Sunday,2,2020-02-02,02:31:56,Sunday,7
4636,19,19,2020-02-02 02:59:48,6.69,5.3,9498,793,1,client,finished,...,0,-316.0,2020,2,Sunday,2,2020-02-02,02:59:48,Sunday,7
4160,19,19,2020-02-02 02:59:48,6.69,5.3,9498,793,1,client,finished,...,0,-316.0,2020,2,Sunday,2,2020-02-02,02:59:48,Sunday,7
4861,26,26,2020-02-02 06:44:20,6000.00,8500.0,12,160,1,client,finished,...,0,NaN,2020,2,Sunday,6,2020-02-02,06:44:20,Sunday,7
2549,26,26,2020-02-02 06:44:20,6000.00,8500.0,12,160,1,client,finished,...,0,NaN,2020,2,Sunday,6,2020-02-02,06:44:20,Sunday,7
2585,26,26,2020-02-02 06:44:20,6000.00,8500.0,12,160,1,client,finished,...,0,NaN,2020,2,Sunday,6,2020-02-02,06:44:20,Sunday,7
362,29,29,2020-02-02 05:57:43,7940.22,7500.0,4869,1122,1,client,finished,...,0,NaN,2020,2,Sunday,5,2020-02-02,05:57:43,Sunday,7


In [ ]:
df_clean[df_clean["order_id_new"] == 3].T 
## .T keerab tabeli lihtsalt teistpidi, et kõiki 30+ veergu oleks lihtne võrrelda. 
## Vaatasin, miks ticket_id_new kordub
## Leidin et customer support ticket ID on siiski erinev, seega võib olla seotud mitu erinevat klienditoe ticket'it.

,1532,1902
order_id_new,3,3
order_try_id_new,3,3
calc_created,2020-02-02 00:49:24,2020-02-02 00:49:24
metered_price,14.87,14.87
upfront_price,NaN,NaN
distance,15541,15541
duration,1690,1690
gps_confidence,0,0
entered_by,client,client
b_state,finished,finished


*Leidsin, et osad order ID'd korduvad, kuna neil on mitu ticketit avatud*

### Millised sõidu olekud andmetes

In [29]:
# Kontrollin, millised sõidu olekud (b_state) andmetes esinevad

df_clean["b_state"].value_counts(dropna=False)

b_state
finished    4943
Name: count, dtype: int64

### Kes ja kui palju sisestas sihtkoha aadressi

In [30]:
# Kontrollin, kes sisestas sihtkoha aadressi ja kui palju iga väärtust esineb

df_clean["entered_by"].value_counts(dropna=False)

entered_by
client      4722
driver       216
reseller       5
Name: count, dtype: int64

### Millised hinnaprognoosi tüübid

In [ ]:
# Kontrollin, millised hinnaprognoosi tüübid andmetes esinevad
# dropna=False näitab tulemuses ka puuduvaid väärtusi (NaN)

df_clean["prediction_price_type"].value_counts(dropna=False)

prediction_price_type
upfront                        3432
prediction                     1279
upfront_destination_changed     208
NaN                              20
upfront_waypoint_changed          4
Name: count, dtype: int64

### Kontrollin tellimuste staatuste väärtusi

In [32]:
# Kontrollin tellimuste staatuste väärtusi

df_clean["order_state"].value_counts(dropna=False)

order_state
finished    4942
active         1
Name: count, dtype: int64

### Kontrollin order try staatuste väärtusi

In [33]:
# Kontrollin order try staatuste väärtusi

df_clean["order_try_state"].value_counts(dropna=False)

order_try_state
finished    4943
Name: count, dtype: int64

### Millisel tellimusel order state active? 

In [34]:
# Vaatan üle ainsa tellimuse, mille order_state ei ole "finished"

df_clean[df_clean["order_state"] == "active"].T

,1296
order_id_new,457
order_try_id_new,457
calc_created,2020-02-06 18:30:35
metered_price,22071.02
upfront_price,12500.0
distance,16986
duration,1547
gps_confidence,0
entered_by,client
b_state,finished


### Kas on sama nime või sisuga veerge?

In [35]:
# Kontrollin, kas andmestikus on sama nimega veerge

df_clean.columns.duplicated().sum()

np.int64(0)

In [36]:
# Kuvan kõik veerunimed, et kontrollida nende nimetusi ja võimalikke ebakõlasid

df_clean.columns.tolist()

['order_id_new',
 'order_try_id_new',
 'calc_created',
 'metered_price',
 'upfront_price',
 'distance',
 'duration',
 'gps_confidence',
 'entered_by',
 'b_state',
 'dest_change_number',
 'prediction_price_type',
 'predicted_distance',
 'predicted_duration',
 'change_reason_pricing',
 'ticket_id_new',
 'rider_app_version',
 'order_state',
 'order_try_state',
 'driver_app_version',
 'driver_device_uid_new',
 'device_name',
 'eu_indicator',
 'overpaid_ride_ticket',
 'fraud_score',
 'year',
 'month',
 'weekday',
 'hour',
 'date',
 'time',
 'day_of_week',
 'day_of_week_nr']

In [ ]:
# Kontrollin, kas erineva nimega veergudes on täpselt sama sisu
# Kui leitakse identsed veerud, kuvab Python nende nimed 
## --> tulemus küll tuli, et on kaks identset veergu, aga kuna need tähendavad erinevat asja, siis jätzme mõlemad alles

for i, col1 in enumerate(df_clean.columns):
    for col2 in df_clean.columns[i + 1:]:
        if df_clean[col1].equals(df_clean[col2]):
            print(col1, "=", col2)

b_state = order_try_state


In [44]:
# Vaatan oluliste numbriliste veergude põhilist statistikat,
# et leida võimalikke ebaloogilisi või äärmuslikke väärtusi

df_clean[
    [
        "metered_price",
        "upfront_price",
        "distance",
        "duration",
        "predicted_distance",
        "predicted_duration",
        "fraud_score"
    ]
].describe()


,metered_price,upfront_price,distance,duration,predicted_distance,predicted_duration,fraud_score
count,4923.000000,3409.000000,4943.000000,4943.000000,4923.000000,4923.000000,2184.000000
mean,7998.471296,4160.095747,9769.223144,1566.230629,8822.636807,1106.737355,-674.046703
std,15815.850352,17015.711912,10912.426401,1650.329858,10548.801733,806.098535,1119.189890
min,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,-14225.000000
25%,5.380000,4.200000,3785.500000,604.000000,4130.500000,597.500000,-826.500000
50%,13.350000,6.600000,7140.000000,1054.000000,6918.000000,939.000000,-278.500000
75%,10991.670000,4000.000000,11953.000000,1929.500000,10674.000000,1427.000000,-64.750000
max,194483.520000,595000.000000,233190.000000,22402.000000,353538.000000,20992.000000,49.000000


In [45]:
# Kontrollin, kas hinnas, distantsis või kestuses esineb
# negatiivseid väärtusi, mis võivad viidata vigastele andmetele

numeric_columns = [
    "metered_price",
    "upfront_price",
    "distance",
    "duration",
    "predicted_distance",
    "predicted_duration"
]

for column in numeric_columns:
    print(column, "negative values:", (df_clean[column] < 0).sum())

metered_price negative values: 0
upfront_price negative values: 0
distance negative values: 0
duration negative values: 0
predicted_distance negative values: 0
predicted_duration negative values: 0


In [47]:
# Kontrollin lõplikult, kas iga veeru andmetüüp on loogiline

df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 4943 entries, 0 to 4942
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   order_id_new           4943 non-null   int64         
 1   order_try_id_new       4943 non-null   int64         
 2   calc_created           4943 non-null   datetime64[us]
 3   metered_price          4923 non-null   float64       
 4   upfront_price          3409 non-null   float64       
 5   distance               4943 non-null   int64         
 6   duration               4943 non-null   int64         
 7   gps_confidence         4943 non-null   int64         
 8   entered_by             4943 non-null   str           
 9   b_state                4943 non-null   str           
 10  dest_change_number     4943 non-null   int64         
 11  prediction_price_type  4923 non-null   str           
 12  predicted_distance     4923 non-null   float64       
 13  predicted_dura

In [49]:
# Kuvan ainult need veerud, kus esineb puuduvaid väärtusi.
# Sorteerin tulemuse nii, et kõige rohkem puuduvate väärtustega veerg on üleval.

missing_values = df_clean.isnull().sum()
missing_values[missing_values > 0].sort_values(ascending=False)


change_reason_pricing    4645
fraud_score              2759
upfront_price            1534
metered_price              20
predicted_distance         20
prediction_price_type      20
predicted_duration         20
rider_app_version          16
dtype: int64

### Kohad kus metered price puudub?

In [51]:
# Kuvan kõik read, kus metered_price puudub,
# koos KÕIGI andmestiku veergudega, et saaksin uurida,
# mis neid 20 kirjet teistest eristab

missing_metered = df_clean[df_clean["metered_price"].isna()]

missing_metered

,order_id_new,order_try_id_new,calc_created,metered_price,upfront_price,distance,duration,gps_confidence,entered_by,b_state,...,device_name,eu_indicator,overpaid_ride_ticket,fraud_score,year,month,date,time,day_of_week,day_of_week_nr
64,217,217,2020-02-04 08:07:10,NaN,NaN,6249,2477,1,driver,finished,...,Xiaomi Redmi 8,1,0,NaN,2020,2,2020-02-04,08:07:10,Tuesday,2
393,3066,3066,2020-03-02 17:45:17,NaN,NaN,5483,917,1,driver,finished,...,HUAWEI EML-L29,1,0,NaN,2020,3,2020-03-02,17:45:17,Monday,1
458,3320,3320,2020-03-06 03:33:52,NaN,NaN,3364,323,0,reseller,finished,...,HUAWEI CLT-L29,1,0,NaN,2020,3,2020-03-06,03:33:52,Friday,5
513,1166,1166,2020-02-13 14:55:14,NaN,NaN,21997,1742,1,driver,finished,...,HUAWEI SLA-L22,1,0,-74.0,2020,2,2020-02-13,14:55:14,Thursday,4
779,1759,1759,2020-02-19 00:55:07,NaN,NaN,10309,849,1,reseller,finished,...,"iPhone8,1",1,0,NaN,2020,2,2020-02-19,00:55:07,Wednesday,3
998,861,861,2020-02-10 06:52:29,NaN,NaN,1424,416,1,driver,finished,...,Samsung SM-G973F,1,0,NaN,2020,2,2020-02-10,06:52:29,Monday,1
1206,1349,1349,2020-02-14 23:08:22,NaN,NaN,11206,1268,1,client,finished,...,Samsung SM-T295,1,0,-1270.0,2020,2,2020-02-14,23:08:22,Friday,5
1287,2012,2012,2020-02-21 13:24:40,NaN,NaN,3717,713,1,driver,finished,...,Samsung SM-A530F,1,0,NaN,2020,2,2020-02-21,13:24:40,Friday,5
1340,1112,1112,2020-02-12 23:32:11,NaN,NaN,4114,528,1,driver,finished,...,Xiaomi Mi A1,1,0,NaN,2020,2,2020-02-12,23:32:11,Wednesday,3
1582,460,460,2020-02-06 19:34:43,NaN,NaN,6718,764,1,client,finished,...,"iPhone8,1",1,0,NaN,2020,2,2020-02-06,19:34:43,Thursday,4


#### Kas leiab veel seoseid metered price'i osas? 

In [52]:
# Uurin 20 rida, kus metered_price puudub.
# Vaatan, kas neil on ühiseid tunnuseid, mis võiksid selgitada puuduvat hinda.

missing_metered[
    [
        "b_state",
        "order_state",
        "order_try_state",
        "prediction_price_type",
        "entered_by",
        "gps_confidence",
        "eu_indicator"
    ]
].value_counts(dropna=False)

b_state   order_state  order_try_state  prediction_price_type  entered_by  gps_confidence  eu_indicator
finished  finished     finished         NaN                    driver      1               1               13
                                                               reseller    1               1                3
                                                                           0               1                2
                                                               client      1               1                2
Name: count, dtype: int64

In [53]:
# Kontrollin, millised olulised väärtused puuduvad
# nendel samadel 20 real

missing_metered.isnull().sum().sort_values(ascending=False)

metered_price            20
upfront_price            20
change_reason_pricing    20
prediction_price_type    20
predicted_distance       20
predicted_duration       20
fraud_score              18
rider_app_version        16
distance                  0
duration                  0
calc_created              0
dest_change_number        0
b_state                   0
order_try_id_new          0
order_id_new              0
gps_confidence            0
entered_by                0
order_state               0
ticket_id_new             0
driver_app_version        0
driver_device_uid_new     0
device_name               0
order_try_state           0
eu_indicator              0
overpaid_ride_ticket      0
year                      0
month                     0
date                      0
time                      0
day_of_week               0
day_of_week_nr            0
dtype: int64

In [54]:
# Vaatan 20 erandliku sõidu device_name väärtusi,
# et kontrollida, kas puuduvad andmed võivad olla seotud
# konkreetse seadme või platvormiga

missing_metered["device_name"].value_counts(dropna=False)

device_name
iPhone8,1            2
Samsung SM-G973F     2
Sony G8441           2
Xiaomi Redmi 8       1
HUAWEI EML-L29       1
HUAWEI CLT-L29       1
HUAWEI SLA-L22       1
Samsung SM-T295      1
Samsung SM-A530F     1
Xiaomi Mi A1         1
Xiaomi Redmi 6A      1
Samsung SM-A705FN    1
CUBOT_P20            1
Samsung SM-G935F     1
HUAWEI MAR-LX1A      1
Samsung SM-J510FN    1
Samsung SM-A405FN    1
Name: count, dtype: int64

In [55]:
# Vaatan nende 20 sõidu rider_app_version väärtusi,
# et kontrollida, kas puuduvad pricing-andmed võivad olla seotud
# konkreetse rakenduse versiooniga

missing_metered["rider_app_version"].value_counts(dropna=False)


rider_app_version
NaN        16
CI.4.17     2
CA.5.26     1
CI.3.22     1
Name: count, dtype: int64

### Vaatan sõite, kus upfront price puudub

In [56]:
# Vaatan ainult neid sõite, kus upfront_price puudub,
# ja kontrollin, millised prediction_price_type väärtused neil esinevad

df_clean[
    df_clean["upfront_price"].isna()
]["prediction_price_type"].value_counts(dropna=False)

prediction_price_type
prediction                     1279
upfront_destination_changed     208
upfront                          23
NaN                              20
upfront_waypoint_changed          4
Name: count, dtype: int64

### Vaatan kus upfront price on puudu

In [57]:
# Vaatan sõite, kus upfront_price ON olemas,
# ja kontrollin nende prediction_price_type väärtusi

df_clean[
    df_clean["upfront_price"].notna()
]["prediction_price_type"].value_counts(dropna=False)

prediction_price_type
upfront    3409
Name: count, dtype: int64

In [61]:
# Uurin 23 sõitu, kus prediction_price_type = "upfront",
# kuid upfront_price puudub.
# Kuvan valitud tunnuste väärtused ja nende esinemissageduse,
# et leida võimalikke ühiseid mustreid.

columns_to_check = [
    "gps_confidence",
    "dest_change_number",
    "change_reason_pricing",
    "entered_by",
    "b_state",
    "rider_app_version",
    "device_name"
]

for column in columns_to_check:
    print("\n---", column, "---")
    print(missing_upfront[column].value_counts(dropna=False))


--- gps_confidence ---
gps_confidence
0    23
Name: count, dtype: int64

--- dest_change_number ---
dest_change_number
1    21
5     2
Name: count, dtype: int64

--- change_reason_pricing ---
change_reason_pricing
NaN    23
Name: count, dtype: int64

--- entered_by ---
entered_by
client    21
driver     2
Name: count, dtype: int64

--- b_state ---
b_state
finished    23
Name: count, dtype: int64

--- rider_app_version ---
rider_app_version
CI.4.19    6
CI.4.17    5
CI.4.18    3
CI.4.22    2
CA.5.42    2
CA.5.43    2
CA.5.44    1
CI.4.00    1
CI.3.91    1
Name: count, dtype: int64

--- device_name ---
device_name
HUAWEI VNS-L21        2
Samsung SM-A530F      2
Samsung SM-A520F      2
Samsung SM-J610FN     2
Coolpad E502          2
HUAWEI ATU-L21        2
HUAWEI PRA-LX1        1
Samsung SM-A202F      1
iPhone9,3             1
Samsung SM-G960F      1
Samsung SM-G973F      1
HUAWEI VNS-L31        1
Samsung SM-N975F      1
iPhone8,1             1
HTC One X10           1
HMD Global TA-1053 